# Forschungsfrage 1 — Modelle auf Colab laufen lassen

Rechnet Random Forest und XGBoost ungetunt über die Kaggle-Playground-Datensätze
und schreibt die Ergebnisse auf Google Drive.


**Vorher einmalig einrichten:**

1. Kaggle-Token unter [kaggle.com/settings](https://www.kaggle.com/settings) →
   *API* → *Create New Token* holen (beginnt mit `KGAT_`).
2. Links in Colab auf das Schlüssel-Symbol (*Secrets*), neues Secret namens
   `KAGGLE_API_TOKEN` anlegen, Token einfügen, *Notebook access* anschalten.
3. Für die GPU-Modelle später: *Laufzeit* → *Laufzeittyp ändern* → GPU.
   Für RandomTree und XGBoost reicht die CPU-Laufzeit.

## 1. Drive einbinden und Ergebnisordner festlegen

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Ordner, in dem die Ergebnis-CSV landet. Aus einem Drive-Link laesst sich der
# Pfad nicht ableiten, der enthaelt nur eine ID - deshalb hier der Name.
ERGEBNIS_ORDNER = Path("/content/drive/MyDrive/Colab Notebooks/output")

if ERGEBNIS_ORDNER.is_dir():
    print("Ergebnisordner gefunden:", ERGEBNIS_ORDNER)
else:
    print("Nicht gefunden:", ERGEBNIS_ORDNER)
    print()
    print("Diese Ordner liegen daneben, den richtigen Namen oben eintragen:")
    eltern = ERGEBNIS_ORDNER.parent
    if eltern.is_dir():
        for p in sorted(eltern.iterdir()):
            if p.is_dir():
                print("   ", p.name)
    else:
        print("auch", eltern, "gibt es nicht - stimmt der Name von MyDrive?")


## 2. Skripte holen

In [ ]:
import os

REPO = "https://github.com/tobiasewald/ProjectBIWise26.git"
ORDNER = Path("/content/ProjectBIWise26")

if ORDNER.is_dir():
    !cd {ORDNER} && git pull --quiet && echo "aktualisiert"
else:
    !git clone --quiet {REPO} {ORDNER} && echo "geklont"

os.chdir(ORDNER)
!ls scripts metadata

Ist das Repository privat, fragt `git clone` nach Zugangsdaten und bleibt
hängen. Dann entweder das Repository öffentlich stellen oder die drei Dateien
`scripts/*.py` plus `metadata/*.csv` von Hand nach Drive legen und von dort
kopieren.

In [ ]:
!pip install --quiet -r requirements.txt
import sklearn, xgboost
print("scikit-learn", sklearn.__version__, "/ xgboost", xgboost.__version__)

Die beiden Versionen müssen `1.7.2` und `3.0.5` sein. Falls `pip` eine Laufzeit-Neustart
verlangt, einmal neu starten und ab hier weiterlaufen.

## 3. Kaggle-Token aus den Secrets

In [ ]:
from google.colab import userdata

# Das Token wird nur in die Umgebung gesetzt, nie ausgegeben und nie im
# Notebook gespeichert. Das Downloadskript sucht dort zuerst.
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
print("Token gesetzt, Laenge", len(os.environ["KAGGLE_API_TOKEN"]))

## 4. Hauptauswertung: alle Modelle auf 10.000 Zeilen

Jeder Datensatz wird von Kaggle geholt, gerechnet und wieder gelöscht. Von klein nach groß, damit bei einem Abbruch
schon Ergebnisse da sind.

Fertige Kombinationen werden beim nächsten Lauf übersprungen. Reißt die Sitzung
ab: Zellen 1 bis 3 erneut ausführen, dann diese. Nichts geht verloren.

In [ ]:
!python scripts/run_models.py --laden --daten-loeschen     --daten /content/data     --ausgabe "{ERGEBNIS_ORDNER}"

## 5. Zwischenstand ansehen

In [ ]:
import pandas as pd

erg = pd.read_csv(ERGEBNIS_ORDNER / "model_results.csv", sep=";", encoding="utf-8-sig")
print(len(erg), "Ergebniszeilen,", erg["slug"].nunique(), "Datensaetze")
print("Umgebungen:", erg["umgebung"].unique())
print("Status:", erg["status"].value_counts().to_dict())

erg[erg["status"] == "ok"].pivot_table(
    index="slug", columns="modell", values="wert_mittel"
).join(erg.set_index("slug")[["metrik_code", "richtung"]].drop_duplicates())

## 6. Zweite Variante: volle Datenmenge

Nur für Wald und Boosting sinnvoll und zeigt, was die Deckelung auf 10.000
Zeilen kostet. S4E7 hat 11,5 Millionen Zeilen. Deshalb ein
eigener Lauf, der die Hauptauswertung nicht blockiert.

In [ ]:
!python scripts/run_models.py --variante voll --laden --daten-loeschen     --daten /content/data     --ausgabe "{ERGEBNIS_ORDNER}"

## 7. TabPFN

Braucht eine **GPU-Laufzeit** (*Laufzeit* → *Laufzeittyp ändern* → GPU). Auf CPU
bricht TabPFN ab einer Grenze von 1.000 Zeilen ab, unsere Variante trainiert auf
8.000.

Verwendet werden die Gewichte **TabPFN-2**. Die sind unter der Prior Labs License
(Apache 2.0 mit Namensnennung) frei ladbar. Die neueren Stände ab TabPFN-2.5
verlangen ein Konto bei Prior Labs, eine im Browser bestätigte Lizenz und einen
API-Key — dann kann nicht mehr jeder in der Gruppe das Projekt einfach
aufsetzen. Mit Key umstellbar über `--tabpfn-version V3_5`.

Der Gewichtsstand steht in jeder Ergebniszeile in der Spalte `modell_version`.

In [ ]:
!pip install --quiet tabpfn
import torch
print("torch", torch.__version__, "/ GPU verfuegbar:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("ACHTUNG: ohne GPU bricht TabPFN bei 8.000 Trainingszeilen ab.")

In [ ]:
!python scripts/run_models.py --modelle tabpfn --laden --daten-loeschen --daten /content/data --ausgabe "{ERGEBNIS_ORDNER}"

## 8. TabFM

**Wichtig: erst ausführen, wenn Wald, XGBoost und TabPFN fertig sind.**

TabFM legt seine Abhängigkeiten fest auf `scikit-learn==1.6.0` und
`torch==2.12.1`. Unsere Ergebnisse für Wald und XGBoost entstehen unter
scikit-learn 1.7.2 — und bei Forschungsfrage 1 sind genau die Voreinstellungen
dieser Bibliothek der Untersuchungsgegenstand. Würde die Installation sie
herunterstufen, hätte ein späterer Lauf andere Zahlen, ohne dass es auffällt.

Deshalb: TabFM bekommt eine eigene Laufzeit. *Laufzeit* → *Laufzeit trennen und
löschen*, dann Zellen 1 bis 3 erneut, dann hier weiter. Die Ergebnisdatei wird
nur angehängt, die vorhandenen Zeilen bleiben erhalten. Die Zelle unten prüft
danach, ob sich die Version verschoben hat.

In [ ]:
!git clone --quiet https://github.com/google-research/tabfm.git /content/tabfm
!cd /content/tabfm && pip install --quiet -e .[pytorch]

import importlib, sklearn, torch
importlib.reload(sklearn)
print("scikit-learn", sklearn.__version__, "/ torch", torch.__version__)
if sklearn.__version__ != "1.7.2":
    print()
    print("ACHTUNG: scikit-learn ist nicht mehr 1.7.2.")
    print("In dieser Laufzeit duerfen rf und xgb nicht mehr gerechnet werden,")
    print("sonst stehen in der Tabelle Zahlen aus zwei Bibliotheksversionen.")

In [ ]:
!python scripts/run_models.py --modelle tabfm --laden --daten-loeschen --daten /content/data --ausgabe "{ERGEBNIS_ORDNER}"

## 9. Auswertung: mittlere Ränge

Fasst die Ergebnisse über alle Datensätze zusammen. Zehn verschiedene Metriken
lassen sich nicht mitteln, Ränge schon: Pro Datensatz Rang 1 für das beste
Modell, darüber der Mittelwert. Dazu der Friedman-Test und der Nemenyi-Nachtest
mit kritischer Differenz (Demšar 2006).

Schreibt `ranks_n10000.csv` und das CD-Diagramm `cd_n10000.png` in denselben
Ordner auf Drive.

In [ ]:
!python scripts/aggregate_ranks.py --ergebnisse "{ERGEBNIS_ORDNER}/model_results.csv" --ausgabe "{ERGEBNIS_ORDNER}"

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ERGEBNIS_ORDNER / "cd_n10000.png")))

Im Diagramm gilt: Modelle unter einem gemeinsamen Balken liegen enger
beieinander als die kritische Differenz — ihr Unterschied ist **nicht belegt**.
Ein Vorsprung in der Rangliste allein reicht als Aussage also nicht.